In [4]:
import pandas as pd
import os

# 🔹 Input file
input_file = r"C:\Users\Shaik Rashed Shariff\OneDrive\Desktop\EXPERIMENT\Main_project\SmartEduCare\static\data\uploads\REF_Share\filter\College_3rdYear_Complete_Data.xlsx"

# 🔹 Output folder
output_dir = r"C:\Users\Shaik Rashed Shariff\OneDrive\Desktop\EXPERIMENT\Main_project\SmartEduCare\static\data\uploads\REF_Share\filter\corrected"
os.makedirs(output_dir, exist_ok=True)

# 🔹 Load data
df = pd.read_excel(input_file)

# 🔹 Base columns (NEVER DELETE THESE)
base_cols = [
    "StudentID", "student_name", "roll_number", "email",
    "course", "year_of_admission", "dept",
    "phone", "address", "PAN_CARD", "AISHE_Code"
]

# 🔹 Subjects mapping
dept_subjects = {
    "CSE": [
        "Design & Analysis of Algorithms", "DBMS", "Operating Systems",
        "Computer Networks", "Software Engineering", "Compiler Design", "Web Technologies"
    ],
    "ECE": [
        "Signals & Systems", "Digital Signal Processing", "Microprocessors",
        "Analog Communications", "Digital Communications", "Control Systems", "VLSI Design"
    ],
    "Mechanical": [
        "Thermodynamics", "Heat & Mass Transfer", "Fluid Mechanics",
        "Machine Design", "Manufacturing Technology", "Vibrations", "Power Plants"
    ],
    "Civil": [
        "Structural Analysis", "RCC Design", "Geotechnical Engineering",
        "Water Resources", "Transportation", "Environmental", "Construction Management"
    ],
    "Electrical": [
        "Power Systems", "Electrical Machines", "Power Electronics",
        "Control Systems", "Protection", "High Voltage", "Distribution"
    ],
    "Chemical": [
        "Chemical Reaction Engineering", "Heat Transfer", "Mass Transfer",
        "Thermodynamics", "Process Control", "Plant Design", "Biochemical Engineering"
    ]
}

# 🔥 Function to remove useless columns
def clean_columns(df):
    cols_to_drop = []

    for col in df.columns:
        if col in base_cols:
            continue  # never remove base columns

        series = df[col]

        # Condition 1: All zero
        if (series == 0).all():
            cols_to_drop.append(col)
            continue

        # Condition 2: Mostly zero (>95%)
        zero_ratio = (series == 0).sum() / len(series)
        if zero_ratio > 0.95:
            cols_to_drop.append(col)
            continue

        # Condition 3: Only 1 unique value
        if series.nunique(dropna=True) <= 1:
            cols_to_drop.append(col)

    return df.drop(columns=cols_to_drop), cols_to_drop


# 🔥 Split by college
college_groups = df.groupby("AISHE_Code")

for college_code, college_df in college_groups:

    output_file = os.path.join(output_dir, f"{college_code}_Processed.xlsx")

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

        for dept, subjects in dept_subjects.items():

            dept_df = college_df[college_df["dept"] == dept]

            if dept_df.empty:
                continue

            # 🔹 Keep required columns
            valid_cols = [col for col in (base_cols + subjects) if col in dept_df.columns]
            dept_df = dept_df[valid_cols]

            # 🔥 CLEAN DATA
            dept_df, removed_cols = clean_columns(dept_df)

            print(f"🧹 {college_code} - {dept} removed: {removed_cols}")

            # 🔹 Save sheet
            dept_df.to_excel(writer, sheet_name=dept[:30], index=False)

    print(f"✅ Created: {output_file}")

print("🎉 All cleaned Excel files created successfully!")

IndexError: At least one sheet must be visible